# 🚲 Yulu Bikes — Hypothesis Testing & Demand Analysis
### Business Case Study | Scaler Data Science & ML Programme

---

**Objective:** Identify the factors that significantly drive demand for shared electric cycles in the Indian market.

**Dataset:** `bike_sharing.csv` — 10,886 hourly rental records (Jan 2011 – Dec 2012)

**Tests Applied:**
- 2-Sample T-Test
- One-Way ANOVA (Season & Weather)
- Chi-Square Test of Independence

**Significance Level:** α = 0.05

---


## 1. Import Libraries

In [ ]:
# Standard libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# Statistical tests
from scipy import stats
from scipy.stats import (ttest_ind, f_oneway, chi2_contingency,
                          shapiro, levene, probplot)

# Display settings
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)

# Plot style
plt.rcParams.update({
    'figure.facecolor': '#FAFAFA',
    'axes.facecolor':   '#FAFAFA',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.labelsize':   10,
    'axes.titlesize':   11,
    'axes.titleweight': 'bold',
    'font.family':      'DejaVu Sans',
})

NAVY   = '#1B3A6B'
TEAL   = '#0E8A8A'
ACCENT = '#F4A261'
GREEN  = '#2A9D8F'
RED    = '#E76F51'
MUTED  = '#6B7280'

print("✅ All libraries imported successfully.")


## 2. Load & Inspect Dataset

In [ ]:
# Load dataset
# If running locally, update path as needed
df = pd.read_csv('bike_sharing.csv', parse_dates=['datetime'])

print("Shape:", df.shape)
print("\nColumn Names:", df.columns.tolist())
print("\nData Types:\n", df.dtypes)


In [ ]:
# Preview first few rows
df.head(10)


In [ ]:
# Check for missing values
missing = df.isnull().sum()
print("Missing Values per Column:")
print(missing[missing > 0] if missing.any() else "✅ No missing values found.")


In [ ]:
# Statistical summary of all columns
df.describe(include='all').round(2)


In [ ]:
# Convert categorical columns to appropriate dtype
cat_cols = ['season', 'holiday', 'workingday', 'weather']
for col in cat_cols:
    df[col] = df[col].astype('category')

# Create readable labels
season_map  = {1: 'Spring', 2: 'Summer', 3: 'Fall', 4: 'Winter'}
weather_map = {1: 'Clear', 2: 'Mist', 3: 'Light Rain/Snow', 4: 'Heavy Rain'}

df['season_label']  = df['season'].astype(int).map(season_map)
df['weather_label'] = df['weather'].astype(int).map(weather_map)

print("✅ Categorical columns converted.")
print("\nValue counts — Season:")
print(df['season_label'].value_counts())
print("\nValue counts — Weather:")
print(df['weather_label'].value_counts())


## 3. Exploratory Data Analysis (EDA)

### 3.1 Univariate Analysis — Continuous Variables

We analyze the distribution of each continuous variable to understand range, skew, and presence of outliers.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('Univariate Analysis — Continuous Variables', fontsize=14, fontweight='bold', y=1.01)

cont_vars = ['temp', 'atemp', 'humidity', 'windspeed', 'casual', 'count']
titles    = ['Temperature (°C)', 'Feeling Temp (°C)', 'Humidity (%)',
             'Wind Speed', 'Casual Rentals', 'Total Rentals (count)']
colors    = [TEAL, TEAL, NAVY, NAVY, GREEN, ACCENT]

for ax, var, title, col in zip(axes.flatten(), cont_vars, titles, colors):
    ax.hist(df[var], bins=35, color=col, alpha=0.85, edgecolor='white', linewidth=0.4)
    mean_val = df[var].mean()
    ax.axvline(mean_val, color=RED, linestyle='--', linewidth=1.5, label=f'Mean = {mean_val:.1f}')
    ax.set_title(title)
    ax.set_xlabel(var)
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('plots/p1_univariate_cont.png', dpi=130, bbox_inches='tight')
plt.show()


**Observations:**
- **Temperature & atemp**: Bimodal distribution — two peaks around 10°C and 25°C reflecting winter and summer seasons.
- **Humidity**: Near-normally distributed, centred around 62%. A few extreme values (0%, 100%) are outliers.
- **Wind Speed**: Strongly right-skewed — most hours are calm; rare gusty events pull the tail right.
- **Casual Rentals**: Heavily right-skewed — most casual use is low-volume; occasional spikes on leisure days.
- **Count (Total Rentals)**: Right-skewed with mean ≈ 191. Not perfectly normal — verified via Q-Q plot below.


### 3.2 Univariate Analysis — Categorical Variables

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle('Univariate Analysis — Categorical Variables', fontsize=14, fontweight='bold')

cat_info = [
    ('season',     {1:'Spring',2:'Summer',3:'Fall',4:'Winter'},   'Season',      [TEAL, ACCENT, GREEN, NAVY]),
    ('holiday',    {0:'Not Holiday',1:'Holiday'},                  'Holiday',     [TEAL, RED]),
    ('workingday', {0:'Non-Working',1:'Working'},                  'Working Day', [NAVY, TEAL]),
    ('weather',    {1:'Clear',2:'Mist',3:'Light Rain',4:'Heavy'}, 'Weather',     [TEAL, NAVY, ACCENT, RED]),
]

for ax, (col, mapping, title, pal) in zip(axes, cat_info):
    counts = df[col].astype(int).value_counts().sort_index()
    labels = [mapping.get(k, str(k)) for k in counts.index]
    bars   = ax.bar(labels, counts.values, color=pal[:len(counts)],
                    edgecolor='white', linewidth=0.5, alpha=0.9)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=20)
    for bar, v in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                f'{v:,}', ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig('plots/p2_univariate_cat.png', dpi=130, bbox_inches='tight')
plt.show()


**Observations:**
- **Season**: Balanced — ~2,700 records per season. No dominant season in the raw data.
- **Holiday**: ~3% holidays (500 records) — heavily imbalanced, expected for hourly data.
- **Working Day**: 68% working days vs 32% non-working.
- **Weather**: Clear dominates (66%), followed by Mist (26%), Light Rain (8%). Heavy Rain has only **1 record** — statistically negligible.


### 3.3 Bivariate Analysis

In [ ]:
# ── Working Day vs Count ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Bivariate: Working Day vs Rental Count', fontsize=13, fontweight='bold')

groups = [df[df['workingday']==0]['count'], df[df['workingday']==1]['count']]
labels = ['Non-Working Day', 'Working Day']

# Box plot
bp = axes[0].boxplot(groups, labels=labels, patch_artist=True,
                     medianprops=dict(color=RED, linewidth=2),
                     whiskerprops=dict(color=NAVY), capprops=dict(color=NAVY))
for patch, col in zip(bp['boxes'], [TEAL, NAVY]):
    patch.set_facecolor(col); patch.set_alpha(0.7)
axes[0].set_title('Box Plot'); axes[0].set_ylabel('Rental Count')

# Violin plot
from scipy.stats import gaussian_kde
for i, (data, label, col) in enumerate(zip(groups, labels, [TEAL, NAVY])):
    kde = gaussian_kde(data)
    x   = np.linspace(data.min(), data.max(), 300)
    axes[1].plot(x, kde(x), color=col, linewidth=2.5, label=label)
    axes[1].fill_between(x, kde(x), alpha=0.15, color=col)
axes[1].set_title('KDE Density'); axes[1].set_xlabel('Count')
axes[1].set_ylabel('Density'); axes[1].legend()

plt.tight_layout()
plt.savefig('plots/p3_bivariate_workingday.png', dpi=130, bbox_inches='tight')
plt.show()

print(f"Non-Working Day mean: {groups[0].mean():.2f}")
print(f"Working Day mean:     {groups[1].mean():.2f}")


**Observation:** Working days (mean ≈ 193) and non-working days (mean ≈ 188) show heavily overlapping distributions. The 4.5-unit gap appears small — a formal t-test will confirm significance.


In [ ]:
# ── Season vs Count ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Bivariate: Season vs Rental Count', fontsize=13, fontweight='bold')

season_order  = ['Spring', 'Summer', 'Fall', 'Winter']
season_means  = df.groupby('season_label')['count'].mean().reindex(season_order)
season_groups = [df[df['season_label']==s]['count'].values for s in season_order]
pal = [TEAL, ACCENT, GREEN, NAVY]

bars = axes[0].bar(season_order, season_means.values, color=pal,
                   edgecolor='white', alpha=0.9)
for bar, v in zip(bars, season_means.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+3,
                 f'{v:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[0].set_title('Mean Rentals by Season'); axes[0].set_ylabel('Mean Count')

bp2 = axes[1].boxplot(season_groups, labels=season_order, patch_artist=True,
                      medianprops=dict(color=RED, linewidth=2))
for patch, col in zip(bp2['boxes'], pal):
    patch.set_facecolor(col); patch.set_alpha(0.75)
axes[1].set_title('Distribution by Season'); axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('plots/p4_bivariate_season.png', dpi=130, bbox_inches='tight')
plt.show()


**Observation:** Clear seasonal pattern — Fall peaks (234.42) while Spring troughs (116.34), roughly half the demand. Summer and Winter fall in between. Outliers are present in all seasons, especially Summer/Fall.


In [ ]:
# ── Weather vs Count ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Bivariate: Weather vs Rental Count', fontsize=13, fontweight='bold')

weather_order  = ['Clear', 'Mist', 'Light Rain/Snow']  # Type 4 (n=1) excluded
weather_means  = df[df['weather_label'].isin(weather_order)].groupby('weather_label')['count'].mean().reindex(weather_order)
weather_groups = [df[df['weather_label']==w]['count'].values for w in weather_order]
wpal = [TEAL, NAVY, ACCENT]

bars2 = axes[0].bar(weather_order, weather_means.values, color=wpal,
                    edgecolor='white', alpha=0.9)
for bar, v in zip(bars2, weather_means.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
                 f'{v:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[0].set_title('Mean Rentals by Weather'); axes[0].set_ylabel('Mean Count')
axes[0].tick_params(axis='x', rotation=10)

bp3 = axes[1].boxplot(weather_groups, labels=weather_order, patch_artist=True,
                      medianprops=dict(color=RED, linewidth=2))
for patch, col in zip(bp3['boxes'], wpal):
    patch.set_facecolor(col); patch.set_alpha(0.75)
axes[1].set_title('Distribution by Weather'); axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=10)

plt.tight_layout()
plt.savefig('plots/p5_bivariate_weather.png', dpi=130, bbox_inches='tight')
plt.show()

print("Note: Weather Type 4 (Heavy Rain) has only n=1 record — excluded from visualization.")


**Observation:** Clear monotonic decline — Clear (205.24) → Mist (178.96) → Light Rain (118.85). A 72% gap between best and worst weather conditions. Weather is a strong operational signal.


In [ ]:
# ── Normality Check ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Normality Check — Rental Count Distribution', fontsize=13, fontweight='bold')

# Histogram
axes[0].hist(df['count'], bins=40, color=TEAL, alpha=0.8, edgecolor='white')
axes[0].axvline(df['count'].mean(), color=RED, linestyle='--', linewidth=1.5, label='Mean')
axes[0].set_title('Histogram of Count'); axes[0].set_xlabel('Count')
axes[0].set_ylabel('Frequency'); axes[0].legend()

# Q-Q plot raw
probplot(df['count'], dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot — Raw Count')
axes[1].get_lines()[0].set(color=TEAL, markersize=2, alpha=0.4)
axes[1].get_lines()[1].set(color=RED, linewidth=2)

# Q-Q plot log-transformed
probplot(np.log1p(df['count']), dist='norm', plot=axes[2])
axes[2].set_title('Q-Q Plot — Log(Count)')
axes[2].get_lines()[0].set(color=GREEN, markersize=2, alpha=0.4)
axes[2].get_lines()[1].set(color=RED, linewidth=2)

plt.tight_layout()
plt.savefig('plots/p6_normality.png', dpi=130, bbox_inches='tight')
plt.show()

print("With n = 10,886, CLT guarantees approximate normality of sample means.")
print("Parametric tests (t-test, ANOVA) are valid regardless of raw distribution shape.")


In [ ]:
# ── Correlation Heatmap ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))
corr_cols = ['temp', 'atemp', 'humidity', 'windspeed', 'casual', 'registered', 'count']
corr = df[corr_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(220, 20, as_cmap=True)

sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap=cmap, ax=ax,
            annot_kws={'size': 10}, vmin=-1, vmax=1,
            linewidths=0.5, linecolor='white')
ax.set_title('Correlation Heatmap — Continuous Variables', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/p7_correlation.png', dpi=130, bbox_inches='tight')
plt.show()


**Key Correlations:**
- **temp ↔ atemp**: r ≈ 0.99 — near-perfect collinearity. Use only one in regression.
- **temp → count**: Moderate positive — warmer weather increases ridership.
- **humidity → count**: Weak negative — higher humidity slightly suppresses demand.
- **windspeed → count**: Near-zero — not a strong standalone predictor.


---
## 4. Hypothesis Testing

> **Significance Level: α = 0.05 for all tests**

---


### 4.1 Test 1 — 2-Sample T-Test
**Question:** Does working day status significantly affect the number of cycles rented?

| | |
|---|---|
| **H₀** | μ(working) = μ(non-working) — No difference in mean rentals |
| **H₁** | μ(working) ≠ μ(non-working) — Significant difference exists |


In [ ]:
# Separate groups
working    = df[df['workingday'] == 1]['count']
nonworking = df[df['workingday'] == 0]['count']

print(f"Working Day    — n={len(working):,}  | mean={working.mean():.2f}  | std={working.std():.2f}")
print(f"Non-Working Day — n={len(nonworking):,} | mean={nonworking.mean():.2f} | std={nonworking.std():.2f}")


In [ ]:
# ── Step 1: Check Equal Variance — Levene's Test ──────────────────────────────
lev_stat, lev_p = levene(working, nonworking)
print("── Levene's Test for Equal Variance ──")
print(f"  Statistic : {lev_stat:.4f}")
print(f"  p-value   : {lev_p:.4f}")
print(f"  Decision  : {'Equal variances assumed (p > 0.05)' if lev_p > 0.05 else 'Unequal variances (p ≤ 0.05)'}")


In [ ]:
# ── Step 2: Run 2-Sample T-Test ───────────────────────────────────────────────
equal_var = lev_p > 0.05
t_stat, t_p = ttest_ind(working, nonworking, equal_var=equal_var)

print("── 2-Sample T-Test ──")
print(f"  t-statistic : {t_stat:.4f}")
print(f"  p-value     : {t_p:.4f}")
print(f"  Alpha       : 0.05")
print()
if t_p < 0.05:
    print("✅ REJECT H₀ — Working day significantly affects rentals (p < 0.05)")
else:
    print("❌ FAIL TO REJECT H₀ — No significant difference in rentals (p ≥ 0.05)")


In [ ]:
# ── Visualization ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Test 1: 2-Sample T-Test — Working Day Effect', fontsize=13, fontweight='bold')

from scipy.stats import gaussian_kde
for data, label, col in [(nonworking,'Non-Working',TEAL),(working,'Working',NAVY)]:
    kde = gaussian_kde(data)
    x   = np.linspace(data.min(), data.max(), 300)
    axes[0].plot(x, kde(x), label=label, color=col, linewidth=2.5)
    axes[0].fill_between(x, kde(x), alpha=0.15, color=col)
axes[0].set_title(f't = {t_stat:.3f}  |  p = {t_p:.4f}')
axes[0].set_xlabel('Rental Count'); axes[0].set_ylabel('Density'); axes[0].legend()

means = [nonworking.mean(), working.mean()]
sems  = [nonworking.sem(), working.sem()]
bars  = axes[1].bar(['Non-Working','Working'], means, yerr=sems,
                    color=[TEAL, NAVY], alpha=0.85, capsize=6, edgecolor='white')
for bar, m in zip(bars, means):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
                 f'{m:.1f}', ha='center', va='bottom', fontweight='bold')
axes[1].set_title('Mean Rentals ± SEM'); axes[1].set_ylabel('Mean Count')

plt.tight_layout()
plt.savefig('plots/p8_ttest.png', dpi=130, bbox_inches='tight')
plt.show()


**Conclusion:** Working day does NOT significantly affect rental demand (p = 0.2264 >> 0.05).
The 4.5-unit mean difference is within natural variation. Yulu's cycles are used comparably
on both weekdays and weekends — serving commuters and leisure riders in balanced proportions.

**Business Inference:** Maintain a uniform 7-day fleet baseline. Optimise for diurnal hourly
peaks rather than day-type splits. Discard complex weekday-only terminal positioning strategies.


---
### 4.2 Test 2a — One-Way ANOVA: Season Effect
**Question:** Does the season significantly affect the number of cycles rented?

| | |
|---|---|
| **H₀** | μSpring = μSummer = μFall = μWinter — Equal means across all seasons |
| **H₁** | At least one season has a significantly different mean rental count |


In [ ]:
# Separate season groups
spring = df[df['season'] == 1]['count'].values
summer = df[df['season'] == 2]['count'].values
fall   = df[df['season'] == 3]['count'].values
winter = df[df['season'] == 4]['count'].values

print("Season Group Statistics:")
for name, g in zip(['Spring','Summer','Fall','Winter'],[spring,summer,fall,winter]):
    print(f"  {name:<8} — n={len(g):,}  mean={g.mean():.2f}  std={g.std():.2f}")


In [ ]:
# ── Step 1: Levene's Test ─────────────────────────────────────────────────────
lev_s, lev_sp = levene(spring, summer, fall, winter)
print("── Levene's Test for Equal Variance ──")
print(f"  Statistic : {lev_s:.4f}")
print(f"  p-value   : {lev_sp:.4f}")
if lev_sp > 0.05:
    print("  → Equal variances assumed ✓")
else:
    print("  → Unequal variances detected. With n ≈ 2,700 per group, ANOVA is robust to this violation.")


In [ ]:
# ── Step 2: One-Way ANOVA ────────────────────────────────────────────────────
f_s, p_s = f_oneway(spring, summer, fall, winter)

print("── One-Way ANOVA — Season ──")
print(f"  F-statistic : {f_s:.4f}")
print(f"  p-value     : {p_s:.6f}")
print(f"  Alpha       : 0.05")
print()
if p_s < 0.05:
    print("✅ REJECT H₀ — Season significantly affects rental demand (p < 0.05)")
else:
    print("❌ FAIL TO REJECT H₀ — No significant seasonal difference (p ≥ 0.05)")


In [ ]:
# ── Visualization ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Test 2a: ANOVA — Season Effect on Rentals', fontsize=13, fontweight='bold')

season_groups = [spring, summer, fall, winter]
season_labels = ['Spring', 'Summer', 'Fall', 'Winter']
pal = [TEAL, ACCENT, GREEN, NAVY]

means_s = [g.mean() for g in season_groups]
axes[0].bar(season_labels, means_s, color=pal, edgecolor='white', alpha=0.9)
for i, (label, m) in enumerate(zip(season_labels, means_s)):
    axes[0].text(i, m+2, f'{m:.1f}', ha='center', fontsize=9, fontweight='bold')
axes[0].set_title(f'Mean Rentals | F={f_s:.2f}, p<0.0001')
axes[0].set_ylabel('Mean Count')

bp = axes[1].boxplot(season_groups, labels=season_labels, patch_artist=True,
                     medianprops=dict(color=RED, linewidth=2))
for patch, col in zip(bp['boxes'], pal):
    patch.set_facecolor(col); patch.set_alpha(0.75)
axes[1].set_title('Distribution by Season'); axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('plots/p9_season_anova.png', dpi=130, bbox_inches='tight')
plt.show()


**Conclusion:** Season has a highly significant effect (F=236.95, p<0.0001).
Fall peaks at 234.42 while Spring troughs at 116.34 — roughly half the demand.

**Business Inference:** Concentrate major preventive fleet maintenance into the low-demand
Spring window to maximise uptime before high-volume Summer and Fall cycles arrive.
Deploy maximum fleet in Fall and Summer.


---
### 4.3 Test 2b — One-Way ANOVA: Weather Effect
**Question:** Does weather condition significantly affect the number of cycles rented?

| | |
|---|---|
| **H₀** | μ₁ = μ₂ = μ₃ — Equal means across weather types |
| **H₁** | At least one weather type has a significantly different mean rental count |

> Note: Weather Type 4 (Heavy Rain) has only n=1 record — excluded from the ANOVA to protect mathematical stability of residual variance.


In [ ]:
# Separate weather groups (exclude Type 4 — n=1)
clear      = df[df['weather'] == 1]['count'].values
mist       = df[df['weather'] == 2]['count'].values
light_rain = df[df['weather'] == 3]['count'].values

print("Weather Group Statistics:")
for name, g in zip(['Clear','Mist','Light Rain/Snow'],[clear,mist,light_rain]):
    print(f"  {name:<18} — n={len(g):,}  mean={g.mean():.2f}  std={g.std():.2f}")

type4 = df[df['weather'] == 4]
print(f"\n  Heavy Rain (Type 4) — n={len(type4)} → Excluded (insufficient data for group comparison)")


In [ ]:
# ── Levene's Test ────────────────────────────────────────────────────────────
lev_w, lev_wp = levene(clear, mist, light_rain)
print("── Levene's Test ──")
print(f"  Statistic : {lev_w:.4f} | p-value : {lev_wp:.4f}")
print("  → Unequal variances. Large sample robustness applies — proceed with note.")

# ── One-Way ANOVA ─────────────────────────────────────────────────────────────
f_w, p_w = f_oneway(clear, mist, light_rain)
print("\n── One-Way ANOVA — Weather ──")
print(f"  F-statistic : {f_w:.4f}")
print(f"  p-value     : {p_w:.6f}")
print()
if p_w < 0.05:
    print("✅ REJECT H₀ — Weather significantly affects rental demand (p < 0.05)")
else:
    print("❌ FAIL TO REJECT H₀ — No significant weather difference (p ≥ 0.05)")


In [ ]:
# ── Visualization ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Test 2b: ANOVA — Weather Effect on Rentals', fontsize=13, fontweight='bold')

weather_groups = [clear, mist, light_rain]
weather_labels = ['Clear', 'Mist', 'Light Rain/Snow']
wpal = [TEAL, NAVY, ACCENT]

means_w = [g.mean() for g in weather_groups]
axes[0].bar(weather_labels, means_w, color=wpal, edgecolor='white', alpha=0.9)
for i, (label, m) in enumerate(zip(weather_labels, means_w)):
    axes[0].text(i, m+2, f'{m:.1f}', ha='center', fontsize=9, fontweight='bold')
axes[0].set_title(f'Mean Rentals | F={f_w:.2f}, p<0.0001')
axes[0].set_ylabel('Mean Count')

bp2 = axes[1].boxplot(weather_groups, labels=weather_labels, patch_artist=True,
                      medianprops=dict(color=RED, linewidth=2))
for patch, col in zip(bp2['boxes'], wpal):
    patch.set_facecolor(col); patch.set_alpha(0.75)
axes[1].set_title('Distribution by Weather'); axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('plots/p10_weather_anova.png', dpi=130, bbox_inches='tight')
plt.show()


**Conclusion:** Weather significantly affects demand (F=65.53, p<0.0001).
Clear weather generates 72% more rentals than Light Rain (205 vs 119 mean).

**Business Inference:** Implement automated app triggers — if Weather Type 3 (Light Rain/Snow)
persists for more than 3 consecutive hours, push ride credits or rainy-day subscription
extensions to active users to smooth monthly churn.


---
### 4.4 Test 3 — Chi-Square Test of Independence
**Question:** Is weather type statistically dependent on season?

| | |
|---|---|
| **H₀** | Weather type and season are independent (no association) |
| **H₁** | Weather type is dependent on season (significant association exists) |


In [ ]:
# Build contingency table
contingency = pd.crosstab(df['season'].astype(int), df['weather'].astype(int),
                           rownames=['Season'], colnames=['Weather'])

# Rename for readability
contingency.index   = ['Spring', 'Summer', 'Fall', 'Winter']
contingency.columns = ['Clear', 'Mist', 'Light Rain', 'Heavy Rain']

print("Contingency Table — Observed Counts:")
print(contingency)
print()
print("Row Percentages (%):")
print((contingency.div(contingency.sum(axis=1), axis=0) * 100).round(1))


In [ ]:
# ── Chi-Square Test ──────────────────────────────────────────────────────────
chi2, p_chi, dof, expected = chi2_contingency(contingency)

print("── Chi-Square Test of Independence ──")
print(f"  Chi2 Statistic : {chi2:.4f}")
print(f"  p-value        : {p_chi:.6f}")
print(f"  Degrees of Freedom : {dof}")
print()

# Check expected frequency assumption
min_expected = expected.min()
print(f"  Min expected frequency: {min_expected:.2f}")
if min_expected < 5:
    print("  ⚠️  Some expected frequencies < 5 (Heavy Rain column). Interpret with caution.")
else:
    print("  ✅ All expected frequencies ≥ 5")

print()
if p_chi < 0.05:
    print("✅ REJECT H₀ — Weather type is dependent on season (p < 0.05)")
else:
    print("❌ FAIL TO REJECT H₀ — Weather and season are independent (p ≥ 0.05)")


In [ ]:
# ── Heatmap Visualization ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Test 3: Chi-Square — Weather vs Season Dependency', fontsize=13, fontweight='bold')

ct_pct = contingency.div(contingency.sum(axis=1), axis=0) * 100

sns.heatmap(contingency, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            linewidths=0.5, linecolor='white', cbar=False)
axes[0].set_title('Observed Counts')

sns.heatmap(ct_pct, annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[1],
            linewidths=0.5, linecolor='white', cbar=False)
axes[1].set_title(f'Row % | χ²={chi2:.2f}, p={p_chi:.4f}')

plt.tight_layout()
plt.savefig('plots/p11_chisquare.png', dpi=130, bbox_inches='tight')
plt.show()


**Conclusion:** Weather and season are NOT independent (χ²=49.16, p<0.0001).
Fall has more Clear weather (1,930 records), Winter has more Mist (807), Spring sees more Light Rain proportionally.

**Business Inference:** Because weather is statistically dependent on season, treating them
as fully independent features in a linear model will introduce multicollinearity.
Future demand forecasting must utilise **Season × Weather interaction terms** to isolate variance cleanly.


---
## 5. Summary & Conclusions

### 5.1 Hypothesis Test Results

| Test | H₀ | Statistic | p-value | Decision |
|------|-----|-----------|---------|----------|
| 2-Sample T-Test (Working Day) | No difference in mean count | t = 1.210 | 0.2264 | ❌ Fail to Reject H₀ |
| ANOVA — Season | Equal means across seasons | F = 236.95 | < 0.0001 | ✅ Reject H₀ |
| ANOVA — Weather | Equal means across weather | F = 65.53 | < 0.0001 | ✅ Reject H₀ |
| Chi-Square (Weather ⊥ Season) | Weather independent of season | χ² = 49.16 | < 0.0001 | ✅ Reject H₀ |

---

### 5.2 Key Findings

1. **Working day does NOT significantly affect demand (p = 0.226).** Yulu serves a balanced mix of commuters and leisure riders. Day-type is not a reliable demand signal.

2. **Season is a major demand driver (F = 236.95, p < 0.0001).** Fall peaks at 234 mean rentals while Spring troughs at 116 — a ~2× gap. The single strongest signal for fleet planning.

3. **Weather significantly suppresses demand (F = 65.53, p < 0.0001).** Clear vs Light Rain shows a 72% usage gap. Weather is a strong, real-time actionable signal.

4. **Weather and season share statistical dependency (χ² = 49.16, p < 0.0001).** Both carry overlapping information — model them with interaction terms.

---

### 5.3 Business Recommendations

**01 · Balanced 7-Day Asset Allocation**
Since the t-test confirms no significant working vs non-working day difference (p = 0.2264), discard complex weekday-only terminal positioning. Maintain a uniform fleet baseline across all 7 days, optimising for diurnal hourly peaks rather than day-type splits.

**02 · Targeted Spring Interventions**
Spring is a sharp demand bottleneck (116.34 mean vs 234.42 in Fall). Concentrate major preventive fleet maintenance into this low-demand Spring window to maximise bike uptime before high-volume Summer and Fall cycles arrive.

**03 · Weather-Triggered Subscription Extensions**
Demand drops 72% from Clear to Light Rain (205 → 119). Implement automated app triggers: if Weather Type 3 persists for more than 3 consecutive hours, push ride credits or rainy-day subscription extensions to smooth monthly churn.

**04 · Interaction-Based Predictive Modelling**
Since weather is statistically dependent on season (χ² = 49.16, p < 0.0001), treating them as independent features introduces multicollinearity. Future demand forecasting must actively use **Season × Weather interaction terms** to isolate variance cleanly.

---

*Analysis performed using Python · pandas · scipy · matplotlib · seaborn*  
*Dataset: 10,886 hourly records | Jan 2011 – Dec 2012 | α = 0.05 throughout*
